# CloudScale AI — Phase 1 Training (Google Colab)

Trains the PPO Reinforcement Learning agent on the K8s+Spot Gymnasium simulator and streams every experiment to your DagsHub MLflow dashboard.

**Before you run:** set your runtime to **GPU → T4** (`Runtime` → `Change runtime type`).

## 1. Set your DagsHub token

Get it from <https://dagshub.com/user/settings/tokens>. Use Colab Secrets (🔑 icon in the left panel):

In [ ]:
from google.colab import userdata
import os
os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')
print('DagsHub token set:', 'YES' if os.environ.get('DAGSHUB_USER_TOKEN') else 'NO — add it to Colab secrets')

## 2. Clone your repo + install deps

**Important:** we deliberately do NOT install torch or numpy — Colab pre-installs them in a way that's binary-compatible with all of Colab's other packages (JAX, TF, etc).

In [ ]:
!git clone https://github.com/bhanujjj/CloudScale-AI-Autonomous-FinOps-Orchestrator.git
%cd CloudScale-AI-Autonomous-FinOps-Orchestrator
!pip install -q -r cloudscale/configs/requirements.txt

## 3. Sanity-check the env (5 random steps)

In [ ]:
from cloudscale.envs.k8s_spot_env import K8sSpotEnv

env = K8sSpotEnv(render_mode='ansi')
obs, info = env.reset(seed=0)
print('obs shape:', obs.shape, '| action space:', env.action_space.n)

for i in range(5):
    a = env.action_space.sample()
    obs, r, term, trunc, info = env.step(a)
    print(f'  step {i}: action={a} reward={r:.4f} terminated={term} truncated={trunc}')

print('env smoke test: PASS')

## 4. Single training run (200k timesteps, ~5 min on T4)

DagsHub will receive every metric, hyperparameter, and the final model artifact.

In [ ]:
!python -m cloudscale.training.train_ppo \
    --mode single \
    --total-timesteps 200000 \
    --eval-episodes 5 \
    --device cuda

## 5. (Optional) Optuna sweep — 20 trials, ~20-30 min on T4

In [ ]:
!python -m cloudscale.training.train_ppo \
    --mode optuna \
    --n-trials 20 \
    --total-timesteps 100000 \
    --device cuda

## 6. Open your DagsHub MLflow dashboard

→ <https://dagshub.com/bhanujbhalla7/CloudScale-AI-Autonomous-FinOps-Orchestrator.mlflow>

You should see a new experiment `cloudscale-ppo-phase1` with all the runs, params, and the saved PPO model artifact.

## 7. (Optional) Evaluate the trained policy

After a `single` run, the model is saved to `cloudscale/models/ppo_seed42.zip`:

In [ ]:
!python -m cloudscale.training.eval_policy \
    --model cloudscale/models/ppo_seed42.zip \
    --episodes 5 \
    --out-dir cloudscale/logs/eval

In [ ]:
from IPython.display import Image, display
import os
for f in sorted(os.listdir('cloudscale/logs/eval')):
    if f.endswith('.png'):
        display(Image(filename=f'cloudscale/logs/eval/{f}'))